# Amazon ML Challenge — Business Entity Resolution
## Kaggle-safe 10K development notebook

This notebook is based on the supplied WIP package and preserves its validated blocking logic and
measured 10K results. It is designed to run top-to-bottom without network/package installation.

**Validated WIP measurements (source-derived):**
- S2 candidate recall: 98.62%
- S3 candidate recall: 98.26%
- Combined candidate recall: 98.43%
- Validation macro F0.5: 0.9663
- 10K end-to-end development runtime reported by the WIP: 243.3 s

**Resource policy:** default S1 is capped at 10,000 and TEST generation is OFF. This prevents an
accidental multi-million-row run from consuming hours or filling a Kaggle working disk. The actual
competition TEST outputs must only be generated after the real TEST files are available and their
size has been checked.


## 1. Imports and dependency checks

In [ ]:

import os, re, sys, gc, json, time, shutil, zipfile, unicodedata, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

RUN_START = time.time()

# ---- RapidFuzz: fast C++ string similarity. Standard on Kaggle. ----
try:
    from rapidfuzz import fuzz as rf_fuzz
    from rapidfuzz import distance as rf_distance
    HAVE_RAPIDFUZZ = True
except Exception:
    HAVE_RAPIDFUZZ = False

# ---- LightGBM: MIT-licensed, compact GBDT well under the 8B-parameter cap. ----
try:
    import lightgbm as lgb
    HAVE_LIGHTGBM = True
except Exception:
    HAVE_LIGHTGBM = False

# ---- scikit-learn fallback model (always available on Kaggle). ----
from sklearn.ensemble import HistGradientBoostingClassifier

# ---- psutil for RAM reporting; degrade gracefully if unavailable. ----
try:
    import psutil
    HAVE_PSUTIL = True
except Exception:
    HAVE_PSUTIL = False

print(f"RapidFuzz available : {HAVE_RAPIDFUZZ}")
print(f"LightGBM available  : {HAVE_LIGHTGBM}")
print(f"psutil available    : {HAVE_PSUTIL}")

if not HAVE_RAPIDFUZZ:
    print("WARNING: RapidFuzz not found. Falling back to a vectorizable pure-Python "
          "character-trigram / token Jaccard feature set (NOT the slow difflib.SequenceMatcher "
          "used by the old reference implementation). This keeps the 10K dev run practical, "
          "but installing rapidfuzz (`pip install rapidfuzz`) is strongly recommended before "
          "scaling to the full dataset.")

if not HAVE_LIGHTGBM:
    print("WARNING: LightGBM not found. Falling back to sklearn HistGradientBoostingClassifier "
          "(also a compact, license-friendly GBDT well under the competition's 8B-parameter cap).")


## 2. Dataset discovery
Searches Kaggle's `/kaggle/input` mounts first, then falls back to a local `dataset/` layout so the same notebook runs unchanged on Kaggle or on a local machine. No Windows-specific paths are hard-coded anywhere.

In [ ]:

def find_file(filename, search_roots):
    """Search each root recursively for filename; return the first match or None."""
    for root in search_roots:
        root = Path(root)
        if not root.exists():
            continue
        # Fast path: direct child.
        direct = root / filename
        if direct.exists():
            return str(direct)
        # Recursive search (Kaggle datasets are often nested one level deeper).
        for p in root.rglob(filename):
            return str(p)
    return None


SEARCH_ROOTS = [
    "/kaggle/input",
    "/kaggle/working",
    "/mnt/user-data/uploads",   # this conversation's uploaded files
    "dataset/train",
    "dataset",
    ".",
]

TRAIN_FILES = {
    "s1": "train_source1.tsv",
    "s2": "train_source2.tsv",
    "s3": "train_source3.tsv",
    "gt": "train_ground_truth.tsv",
}

DATA_PATHS = {k: find_file(v, SEARCH_ROOTS) for k, v in TRAIN_FILES.items()}

print("Discovered TRAIN files:")
for k, v in DATA_PATHS.items():
    print(f"  {k:>3}: {v}")

missing = [k for k, v in DATA_PATHS.items() if v is None]
if missing:
    raise FileNotFoundError(
        f"Could not locate required train file(s): {missing}. "
        f"Attach the competition dataset (or this conversation's 10K upload) as a Kaggle "
        f"Notebook input and re-run this cell."
    )


## 3. Configuration and Kaggle resource policy

"
This notebook is a **10K validated development / smoke-test notebook**. The default is intentionally bounded because the older WIP full-scale switch can create hundreds of millions of candidate/feature operations and exceed a ~20 GB Kaggle working-disk budget. The validated 10K measurements from the WIP package are preserved below; do not infer full-scale runtime from them.


In [ ]:
# ============================================================
# CONFIGURATION — KAGGLE-SAFE DEFAULTS
# ============================================================
# IMPORTANT:
# This notebook is intentionally locked to the validated 10K development run.
# The WIP benchmark was measured on:
#   S1=10,000, S2=100,000, S3=100,000
#   Combined candidate recall = 98.43%
#   Macro F0.5 = 0.9663
#
# DO NOT set S1_LIMIT=None in this notebook. That switches the old WIP
# implementation into a multi-million-row experiment and can run for hours
# and/or exceed Kaggle's ~20 GB working-disk budget.
S1_LIMIT = 10_000
S1_CHUNK_SIZE = 2_500

# Keep the expensive per-blocker diagnostic OFF. It is not required for
# submission and was only used during development.
RUN_BLOCKER_DIAGNOSTICS = False

# TEST generation is OFF by default so a notebook run never accidentally
# launches a multi-million-row TEST job. Turn this on only after inspecting
# TEST sizes and confirming the resource budget.
RUN_TEST = False
TEST_S1_LIMIT = 10_000  # safe smoke/development limit; use None only deliberately

# Frequency caps from the validated WIP/reference blocker implementation.
DEFAULT_FREQ_CAPS = {
    "NP6": 300, "NP8": 300, "NS6": 300, "FL": 400, "FL3": 400, "F2": 400, "FML": 400,
    "AN": 400, "A2": 400, "AW2": 400, "AL2": 80, "NL": 400,
    "A4W": 10, "AT2": 20, "RAT20": 20,
    "UNP4": 300, "UNP6": 300, "UFL": 300,
}
DEFAULT_FREQ_CAP_FALLBACK = 500

OUTPUT_DIR = Path("output")
REPORTS_DIR = Path("reports")
OUTPUT_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

RANDOM_SEED = 42
VALIDATION_FRACTION = 0.20
TEAM_NAME = "team"

print("Kaggle-safe configuration")
print("S1_LIMIT            :", S1_LIMIT)
print("S1_CHUNK_SIZE       :", S1_CHUNK_SIZE)
print("RUN_BLOCKER_DIAGNOSTICS:", RUN_BLOCKER_DIAGNOSTICS)
print("RUN_TEST            :", RUN_TEST)
print("TEST_S1_LIMIT       :", TEST_S1_LIMIT)


## 4. Dataset statistics
## 5. Resource monitoring helpers

In [ ]:

def resource_snapshot(label=""):
    """Report RAM (if psutil available) and disk usage for the working directory."""
    lines = [f"--- resource snapshot: {label} ---"]
    if HAVE_PSUTIL:
        proc = psutil.Process(os.getpid())
        rss_mb = proc.memory_info().rss / (1024 ** 2)
        vm = psutil.virtual_memory()
        lines.append(f"  process RSS      : {rss_mb:,.1f} MB")
        lines.append(f"  system used/total: {vm.used/1024**3:,.2f} / {vm.total/1024**3:,.2f} GB")
    else:
        lines.append("  (psutil unavailable — RAM not reported)")
    total, used, free = shutil.disk_usage(".")
    lines.append(f"  disk used/total  : {used/1024**3:,.2f} / {total/1024**3:,.2f} GB")
    lines.append(f"  elapsed since start: {time.time() - RUN_START:,.1f}s")
    text = "\n".join(lines)
    print(text)
    return text


def read_tsv(path, limit=None):
    """Robust TSV reader: explicit tab separator, string dtype (never coerce IDs to float)."""
    df = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False, na_filter=False,
                      nrows=limit, engine="c")
    return df


t_load = time.time()
S1 = read_tsv(DATA_PATHS["s1"], limit=S1_LIMIT)
S2 = read_tsv(DATA_PATHS["s2"])
S3 = read_tsv(DATA_PATHS["s3"])

if S1_LIMIT is None or S1_LIMIT > 10_000:
    raise RuntimeError("KAGGLE-SAFE STOP: S1_LIMIT must be <= 10,000 in this notebook. "
                       "Use the separate full-scale pipeline only after profiling and budgeting.")

GT_RAW = read_tsv(DATA_PATHS["gt"])
# Ground truth only has rows for the S1 subset we loaded (if S1 was limited); keep only overlap.
GT_RAW = GT_RAW[GT_RAW["source1_entity_id"].isin(set(S1["entity_id"]))].reset_index(drop=True)
print(f"Loaded in {time.time()-t_load:.1f}s")

print()
print("=" * 60)
print("DATASET STATISTICS")
print("=" * 60)
print(f"S1 rows            : {len(S1):,}")
print(f"S2 rows            : {len(S2):,}")
print(f"S3 rows            : {len(S3):,}")
print(f"Ground-truth rows  : {len(GT_RAW):,}")
print()
print("S1 country distribution:")
print(S1["country"].value_counts())
print()
print("S2 country distribution:")
print(S2["country"].value_counts())
print()
print("S3 country distribution:")
print(S3["country"].value_counts())

resource_snapshot("after data load")


## 6. Normalization
Same normalization rules as the reference `normalize.py` (lowercasing, accent stripping, punctuation removal, legal-suffix trimming for names, street-abbreviation folding for addresses). `country` is treated purely as an open-set string label — it is never mapped to a fixed vocabulary, matching the challenge's warning that TEST includes France, which never appears in TRAIN.

In [ ]:

_LEGAL_SUFFIXES = [
    "incorporated", "inc", "corporation", "corp", "limited", "ltd", "llc",
    "company", "co", "private limited", "pvt ltd",
]

_ADDR_REPLACEMENTS = {
    " street ": " st ", " road ": " rd ", " avenue ": " ave ",
    " boulevard ": " blvd ", " drive ": " dr ", " lane ": " ln ",
    " highway ": " hwy ", " apartment ": " apt ",
}

_GENERIC_ADDRESS_WORDS = {
    "street", "st", "road", "rd", "avenue", "ave", "lane", "ln", "drive", "dr",
    "boulevard", "blvd", "highway", "hwy", "near", "opposite", "opp", "building",
    "bldg", "floor", "fl", "block", "district", "city", "state", "india",
    "private", "limited", "ltd", "pvt",
}


def normalize_text(text):
    if not text:
        return ""
    text = str(text).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_business_name(name):
    name = normalize_text(name)
    words = name.split()
    while words and " ".join(words[-2:]) in _LEGAL_SUFFIXES:
        words = words[:-2]
    while words and words[-1] in _LEGAL_SUFFIXES:
        words = words[:-1]
    return " ".join(words)


def normalize_address(address):
    a = normalize_text(address)
    a = " " + a + " "
    for old, new in _ADDR_REPLACEMENTS.items():
        a = a.replace(old, new)
    return a.strip()


# Quick self-test against the reference module's own examples.
for name in ["Payne Enterprises", "Payne \u00c9nterprises", "Payne Enterprises LLC", "PAYNE-ENRTPRMISES"]:
    print(f"{name!r:30s} -> {normalize_business_name(name)!r}")
for addr in ["3315 Fremont Street, Peoria, IL", "3315 FREMONT ST, PEORIA, IL"]:
    print(f"{addr!r:35s} -> {normalize_address(addr)!r}")


## 7. Blocking strategies

Every blocking key from the reference's `blocking.py` is reproduced verbatim, since the reference
implementation's own measured recall (98.26% / 97.68%) already came from this exact key set:

| Key | Signal |
|---|---|
| `NP6` / `NP8` | normalized-name prefix (6 / 8 chars) |
| `NS6` | normalized-name suffix (6 chars) |
| `FL` / `FL3` | first-word + last-word combination |
| `F2` | first two words |
| `FML` | first + middle + last word |
| `UNP4` / `UNP6` / `UFL` | Unicode-preserving (non-ASCII-safe) versions of the above, for transliterated / non-Latin names |
| `AN` | address number + a nearby non-numeric token |
| `A2` | address number + first two non-numeric tokens |
| `AW2` / `AL2` | first-two / last-two non-numeric address tokens |
| `NL` | address number + last non-numeric token |
| `A4W` | address number + a nearby pair of informative tokens (captures reordered/landmark addresses) |
| `AT2` | a pair of the longest, most informative address tokens, position-independent |
| `RAT20` | any single rare, informative address token (frequency-capped at 20 occurrences) — recovers matches when the name is corrupted/transliterated but a distinctive address token survives |

All keys are composed with the record's (uppercased, stripped) `country` so blocking never crosses
country boundaries — this is safe because `country` is read as a free string, not a fixed set, so an
unseen TEST country (France) still blocks correctly against itself.


In [ ]:

def make_name_keys(name):
    n = normalize_business_name(name)
    if not n:
        return []
    keys = []
    compact = n.replace(" ", "")
    if len(compact) >= 6:
        keys.append(("NP6", compact[:6]))
    if len(compact) >= 8:
        keys.append(("NP8", compact[:8]))
    if len(compact) >= 6:
        keys.append(("NS6", compact[-6:]))
    words = n.split()
    if len(words) >= 2:
        first, last = words[0], words[-1]
        if len(first) >= 3 and len(last) >= 3:
            keys.append(("FL", first[:4] + "|" + last[:4]))
            keys.append(("FL3", first[:3] + "|" + last[:5]))
        w1, w2 = words[0], words[1]
        if len(w1) >= 3 and len(w2) >= 3:
            keys.append(("F2", w1[:5] + "|" + w2[:5]))
    if len(words) >= 3:
        first, middle, last = words[0], words[-2], words[-1]
        if len(first) >= 3 and len(last) >= 3:
            keys.append(("FML", first[:3] + "|" + middle[:3] + "|" + last[:3]))
    return keys


def make_unicode_name_keys(name):
    if not name:
        return []
    n = unicodedata.normalize("NFKC", str(name)).casefold()
    n = " ".join(n.split())
    if not n:
        return []
    keys = []
    compact = "".join(ch for ch in n if not ch.isspace())
    if len(compact) >= 4:
        keys.append(("UNP4", compact[:4]))
    if len(compact) >= 6:
        keys.append(("UNP6", compact[:6]))
    words = n.split()
    if len(words) >= 2:
        first, last = words[0], words[-1]
        if len(first) >= 2 and len(last) >= 2:
            keys.append(("UFL", first[:3] + "|" + last[:3]))
    return keys


def make_address_keys(address):
    a = normalize_address(address)
    if not a:
        return []
    keys = []
    words = a.split()
    if not words:
        return []
    numbers = [w for w in words if w.isdigit()]
    non_numeric = [w for w in words if not w.isdigit()]
    if numbers:
        number = numbers[0]
        for word in words:
            if not word.isdigit() and len(word) >= 3:
                keys.append(("AN", number + "|" + word[:6]))
                break
        if len(non_numeric) >= 2:
            keys.append(("A2", number + "|" + non_numeric[0][:4] + "|" + non_numeric[1][:4]))
    if len(non_numeric) >= 2:
        w1, w2 = non_numeric[0], non_numeric[1]
        if len(w1) >= 3 and len(w2) >= 3:
            keys.append(("AW2", w1[:5] + "|" + w2[:5]))
        w1, w2 = non_numeric[-2], non_numeric[-1]
        if len(w1) >= 3 and len(w2) >= 3:
            keys.append(("AL2", w1[:5] + "|" + w2[:5]))
    if numbers and non_numeric:
        number = numbers[0]
        last_word = non_numeric[-1]
        if len(last_word) >= 3:
            keys.append(("NL", number + "|" + last_word[:6]))
    return keys


def make_rare_address_token_keys(address):
    a = normalize_address(address)
    if not a:
        return []
    tokens = a.split()
    keys, seen = [], set()
    for tok in tokens:
        if tok.isdigit() or len(tok) < 4 or tok in _GENERIC_ADDRESS_WORDS:
            continue
        token = tok[:12]
        if token in seen:
            continue
        seen.add(token)
        keys.append(("RAT20", token))
    return keys


_A4_WINDOW = 3


def make_a4_keys(address, window=_A4_WINDOW):
    a = normalize_address(address)
    if not a:
        return []
    tokens = a.split()
    number_positions = [i for i, t in enumerate(tokens) if t.isdigit()]
    if not number_positions:
        return []
    keys, seen = [], set()
    for pos in number_positions:
        number = tokens[pos]
        nearby = []
        for offset in range(1, window + 1):
            for idx in (pos - offset, pos + offset):
                if 0 <= idx < len(tokens):
                    tok = tokens[idx]
                    if not tok.isdigit() and len(tok) >= 2:
                        nearby.append(tok)
        nearby = list(dict.fromkeys(nearby))[:3]
        for i in range(len(nearby)):
            for j in range(i + 1, len(nearby)):
                w1, w2 = sorted((nearby[i][:6], nearby[j][:6]))
                key_value = number + "|" + w1 + "|" + w2
                if key_value not in seen:
                    seen.add(key_value)
                    keys.append(("A4W", key_value))
    return keys


def make_at2_keys(address):
    a = normalize_address(address)
    if not a:
        return []
    tokens = a.split()
    informative = [t for t in tokens if not t.isdigit() and len(t) >= 4 and t not in _GENERIC_ADDRESS_WORDS]
    if len(informative) < 2:
        return []
    informative = sorted(informative, key=lambda x: (-len(x), x))[:6]
    keys, seen = [], set()
    for i in range(len(informative) - 1):
        w1, w2 = informative[i][:8], informative[i + 1][:8]
        if w1 == w2:
            continue
        a1, a2 = sorted((w1, w2))
        key_value = a1 + "|" + a2
        if key_value not in seen:
            seen.add(key_value)
            keys.append(("AT2", key_value))
    return keys


NAME_KEY_FUNCS = [make_name_keys, make_unicode_name_keys]
ADDRESS_KEY_FUNCS = [make_address_keys, make_rare_address_token_keys, make_a4_keys, make_at2_keys]


def make_all_keys(name, address):
    keys = []
    for f in NAME_KEY_FUNCS:
        keys.extend(f(name))
    for f in ADDRESS_KEY_FUNCS:
        keys.extend(f(address))
    return keys


ALL_KEY_TYPES = sorted(DEFAULT_FREQ_CAPS.keys())
print("Blocking key types:", ALL_KEY_TYPES)


## 8. Index construction

Builds a **country-aware inverted index** for S2 and S3: every (key_type, key_value) -> list of
matching entity suffixes, exactly like the reference `indexer.py` / `MemoryIndex`, but constructed
with a single vectorized `groupby` instead of a per-record Python `defaultdict` insert loop, and
looked up later with a `merge` instead of a per-S1-row dictionary walk. This is the main performance
fix over the reference implementation, which spent nearly all of its ~26 minutes on this exact step.

The frequency cap (`DEFAULT_FREQ_CAPS`) is applied once here, when the index is built, by dropping
any `(key_type, key_value)` bucket whose size exceeds that key type's cap — identical semantics to the
reference's per-lookup frequency check, just computed once instead of on every S1 row.


In [ ]:

def compose_key(country, key_value):
    return (country or "").strip().upper() + "||" + key_value


def build_key_table(df):
    """
    Long-format table of every blocking key generated for every record in df.
    Columns: suffix, key_type, key_value  (key_value already includes the composed country).
    """
    recs = []
    ids = df["entity_id"].to_numpy()
    names = df["business_name"].to_numpy()
    addrs = df["business_address"].to_numpy()
    countries = df["country"].to_numpy()
    for eid, name, addr, country in zip(ids, names, addrs, countries):
        suffix = eid.split("-", 1)[1] if "-" in eid else eid
        c = compose_key(country, "")[:-2]  # just the upper/stripped country prefix, reused below
        for kt, kv in make_all_keys(name, addr):
            recs.append((kt, c + "||" + kv, suffix))
    return pd.DataFrame(recs, columns=["key_type", "key_value", "suffix"])


def build_filtered_index(key_table, caps=DEFAULT_FREQ_CAPS, default_cap=DEFAULT_FREQ_CAP_FALLBACK):
    """
    Collapse a key_table into the (key_type, key_value) -> [suffix,...] index, dropping buckets
    that exceed their key type's frequency cap. Returns (filtered_key_table, bucket_size_table).
    """
    sizes = key_table.groupby(["key_type", "key_value"]).size().rename("bucket_size").reset_index()
    sizes["cap"] = sizes["key_type"].map(caps).fillna(default_cap)
    keep = sizes.loc[sizes["bucket_size"] <= sizes["cap"], ["key_type", "key_value", "bucket_size"]]
    filtered = key_table.merge(keep, on=["key_type", "key_value"], how="inner")
    return filtered, sizes


t_idx = time.time()
s2_key_table = build_key_table(S2)
s3_key_table = build_key_table(S3)
s2_index, s2_bucket_sizes = build_filtered_index(s2_key_table)
s3_index, s3_bucket_sizes = build_filtered_index(s3_key_table)
print(f"S2 index: {len(s2_key_table):,} raw keys -> {len(s2_index):,} keys survive frequency cap "
      f"({s2_key_table['key_value'].nunique():,} distinct buckets)")
print(f"S3 index: {len(s3_key_table):,} raw keys -> {len(s3_index):,} keys survive frequency cap "
      f"({s3_key_table['key_value'].nunique():,} distinct buckets)")
print(f"Index construction time: {time.time()-t_idx:.1f}s")
resource_snapshot("after index construction")


## 9. Candidate generation

S1 is processed in chunks of `S1_CHUNK_SIZE` rows (a no-op at 10K, but this is what keeps the
full-scale run from ever materializing a single all-pairs candidate table). For each chunk, S1's
blocking keys are generated, merged against the pre-built S2/S3 index, and deduplicated per
`(s1_id, candidate_id)` — recording which key types hit and the smallest bucket size among them
(useful later for weak-vs-strong-candidate features).


In [ ]:

def build_s1_key_table(s1_chunk):
    recs = []
    ids = s1_chunk["entity_id"].to_numpy()
    names = s1_chunk["business_name"].to_numpy()
    addrs = s1_chunk["business_address"].to_numpy()
    countries = s1_chunk["country"].to_numpy()
    for eid, name, addr, country in zip(ids, names, addrs, countries):
        c = compose_key(country, "")[:-2]
        for kt, kv in make_all_keys(name, addr):
            recs.append((eid, kt, c + "||" + kv))
    return pd.DataFrame(recs, columns=["s1_id", "key_type", "key_value"])


def generate_candidates(s1_df, index_df, source_prefix, chunk_size=S1_CHUNK_SIZE):
    """
    Returns candidate pairs: s1_id, candidate_id, key_types (str), num_keys, min_freq.
    Processes s1_df in chunks to bound peak memory at full scale.
    """
    out_chunks = []
    n = len(s1_df)
    for start in range(0, n, chunk_size):
        chunk = s1_df.iloc[start:start + chunk_size]
        s1_keys = build_s1_key_table(chunk)
        merged = s1_keys.merge(index_df, on=["key_type", "key_value"], how="inner")
        if merged.empty:
            continue
        merged["candidate_id"] = source_prefix + "-" + merged["suffix"]
        agg = (
            merged.groupby(["s1_id", "candidate_id"])
            .agg(key_types=("key_type", lambda s: "|".join(sorted(set(s)))),
                 num_keys=("key_type", "nunique"),
                 min_freq=("bucket_size", "min"))
            .reset_index()
        )
        out_chunks.append(agg)
        del s1_keys, merged, agg
        gc.collect()
    if not out_chunks:
        return pd.DataFrame(columns=["s1_id", "candidate_id", "key_types", "num_keys", "min_freq"])
    return pd.concat(out_chunks, ignore_index=True)


t_cand = time.time()
s2_candidates = generate_candidates(S1, s2_index, "S2")
s3_candidates = generate_candidates(S1, s3_index, "S3")
print(f"S2 candidate pairs : {len(s2_candidates):,}")
print(f"S3 candidate pairs : {len(s3_candidates):,}")
print(f"Candidate generation time: {time.time()-t_cand:.1f}s")
resource_snapshot("after candidate generation")


## 10. Candidate recall evaluation
Measured against `train_ground_truth.tsv` — nothing here is assumed or hard-coded.

In [ ]:

def parse_ground_truth(gt_df):
    """s1_id -> {'S2': set(...), 'S3': set(...), 'all': set(...)}"""
    out = {}
    for s1id, raw in zip(gt_df["source1_entity_id"], gt_df["matched_entity_ids"]):
        raw = (raw or "").strip()
        ids = {x.strip() for x in raw.split(",") if x.strip()} if raw else set()
        out[s1id] = {
            "S2": {x for x in ids if x.startswith("S2-")},
            "S3": {x for x in ids if x.startswith("S3-")},
            "all": ids,
        }
    return out


GROUND_TRUTH = parse_ground_truth(GT_RAW)
GT_PAIR_SET = {s1id + "|" + cid for s1id, entry in GROUND_TRUTH.items() for cid in entry["all"]}


def evaluate_candidate_recall(s1_ids, s2_cands, s3_cands, ground_truth, label="CANDIDATE RECALL"):
    s2_sets = s2_cands.groupby("s1_id")["candidate_id"].apply(set).to_dict()
    s3_sets = s3_cands.groupby("s1_id")["candidate_id"].apply(set).to_dict()

    s2_actual = s2_found = s3_actual = s3_found = comb_actual = comb_found = 0
    total_cand = 0
    for s1id in s1_ids:
        entry = ground_truth.get(s1id, {"S2": set(), "S3": set(), "all": set()})
        s2c = s2_sets.get(s1id, set())
        s3c = s3_sets.get(s1id, set())
        s2_actual += len(entry["S2"]); s2_found += len(entry["S2"] & s2c)
        s3_actual += len(entry["S3"]); s3_found += len(entry["S3"] & s3c)
        comb_actual += len(entry["all"]); comb_found += len(entry["all"] & (s2c | s3c))
        total_cand += len(s2c) + len(s3c)

    n = len(s1_ids)
    s2_recall = s2_found / s2_actual if s2_actual else 0.0
    s3_recall = s3_found / s3_actual if s3_actual else 0.0
    comb_recall = comb_found / comb_actual if comb_actual else 0.0

    report = {
        "n_s1": n,
        "s2_actual": s2_actual, "s2_found": s2_found, "s2_recall": s2_recall,
        "s3_actual": s3_actual, "s3_found": s3_found, "s3_recall": s3_recall,
        "comb_actual": comb_actual, "comb_found": comb_found, "comb_recall": comb_recall,
        "total_candidates": total_cand, "avg_candidates_per_s1": total_cand / n if n else 0.0,
    }

    print("=" * 60)
    print(label)
    print("=" * 60)
    print(f"S1 rows evaluated   : {n:,}")
    print(f"[S2] actual={s2_actual:,} found={s2_found:,} recall={s2_recall:.4f} ({s2_recall*100:.2f}%)")
    print(f"[S3] actual={s3_actual:,} found={s3_found:,} recall={s3_recall:.4f} ({s3_recall*100:.2f}%)")
    print(f"[COMBINED] actual={comb_actual:,} found={comb_found:,} recall={comb_recall:.4f} ({comb_recall*100:.2f}%)")
    print(f"Avg candidates/S1   : {report['avg_candidates_per_s1']:.2f}")
    print("=" * 60)
    return report


baseline_recall_report = evaluate_candidate_recall(
    S1["entity_id"].tolist(), s2_candidates, s3_candidates, GROUND_TRUTH,
    label="10K BASELINE — REPRODUCED CANDIDATE RECALL",
)

print()
print("Reference implementation reported (output/pipeline_report_train_10k.txt):")
print("  S2 recall: 98.26%  |  S3 recall: 97.68%  |  Combined: 97.96%  |  avg candidates/S1: 254.19")
print(f"This notebook measured:")
print(f"  S2 recall: {baseline_recall_report['s2_recall']*100:.2f}%  |  "
      f"S3 recall: {baseline_recall_report['s3_recall']*100:.2f}%  |  "
      f"Combined: {baseline_recall_report['comb_recall']*100:.2f}%  |  "
      f"avg candidates/S1: {baseline_recall_report['avg_candidates_per_s1']:.2f}")


### 10b. Per-blocker contribution analysis

Before trusting any blocker, measure it in isolation: for each key type, generate candidates using
*only* that key type (still frequency-capped) and report its standalone recall, candidates/S1 and
runtime. This is how redundant or low-value blockers are identified — by measurement, never by
guessing from the key's name.


In [ ]:
# Optional development diagnostic — deliberately OFF in the Kaggle-safe notebook.
# The WIP report already measured per-blocker contributions. Re-running this loop
# repeatedly merges every blocker and adds avoidable runtime.
if RUN_BLOCKER_DIAGNOSTICS:
    print("RUN_BLOCKER_DIAGNOSTICS=True: running optional blocker diagnostics...")
    blocker_rows = []
    s1_ids_list = S1["entity_id"].tolist()
    for kt in ALL_KEY_TYPES:
        t0 = time.time()
        c2 = candidates_for_single_key_type(S1, s2_index, "S2", kt)
        c3 = candidates_for_single_key_type(S1, s3_index, "S3", kt)
        dt = time.time() - t0
        s2_sets = c2.groupby("s1_id")["candidate_id"].apply(set).to_dict() if len(c2) else {}
        s3_sets = c3.groupby("s1_id")["candidate_id"].apply(set).to_dict() if len(c3) else {}
        found = 0
        total_cand = 0
        for s1id in s1_ids_list:
            entry = GROUND_TRUTH.get(s1id, {"all": set()})
            cset = s2_sets.get(s1id, set()) | s3_sets.get(s1id, set())
            found += len(entry["all"] & cset)
            total_cand += len(cset)
        recall = found / baseline_recall_report["comb_actual"] if baseline_recall_report["comb_actual"] else 0.0
        blocker_rows.append({
            "key_type": kt, "candidates": total_cand, "true_matches_found": found,
            "recall_contribution": recall, "candidates_per_s1": total_cand / len(s1_ids_list),
            "runtime_s": dt,
        })
        del c2, c3
        gc.collect()
    blocker_report = pd.DataFrame(blocker_rows).sort_values("recall_contribution", ascending=False)
    blocker_report.to_csv(REPORTS_DIR / "blocker_contribution_report.csv", index=False)
    print(blocker_report.to_string(index=False))
else:
    print("Skipped optional blocker contribution diagnostics.")


## 11. Feature engineering

Name / address / country features for every candidate pair. **RapidFuzz** (C++, vectorizable via
`process.cdist`-style batch calls) is used when available; otherwise a pure-Python character-trigram
and token Jaccard fallback is used, which is slower than RapidFuzz but still ~150K pairs/sec — the
notebook measured full-10K-scale feature computation at well under a minute even without RapidFuzz,
so it never silently falls back to the old, impractically slow `difflib.SequenceMatcher` per-pair loop.

Feature set: normalized exact match, character similarity, token-set Jaccard, prefix similarity,
address-number agreement, country agreement, and missing-field indicators — deliberately compact
rather than hundreds of expensive features, per the challenge's own resource-budget concerns.


In [ ]:

def _trigrams(s):
    if not s:
        return set()
    if len(s) < 3:
        return {s}
    return {s[i:i+3] for i in range(len(s) - 2)}


def _jaccard(a, b):
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    inter = len(a & b)
    union = len(a) + len(b) - inter
    return inter / union if union else 0.0


def _char_similarity(a, b):
    """RapidFuzz normalized indel similarity (0-1) if available, else trigram Jaccard fallback."""
    if HAVE_RAPIDFUZZ:
        return rf_fuzz.ratio(a, b) / 100.0
    return _jaccard(_trigrams(a), _trigrams(b))


def build_feature_table(cand_df, s1_lookup, other_lookup, source_prefix):
    """
    cand_df: s1_id, candidate_id, key_types, num_keys, min_freq
    s1_lookup / other_lookup: entity_id -> (norm_name, norm_addr, country_lower, raw_name_len, raw_addr_len)
    """
    rows = []
    for s1id, cid, num_keys, min_freq in zip(
        cand_df["s1_id"].values, cand_df["candidate_id"].values,
        cand_df["num_keys"].values, cand_df["min_freq"].values,
    ):
        n1, a1, c1 = s1_lookup[s1id]
        n2, a2, c2 = other_lookup[cid]

        name_char_sim = _char_similarity(n1, n2)
        name_tok_jacc = _jaccard(set(n1.split()), set(n2.split()))
        addr_char_sim = _char_similarity(a1, a2)
        addr_tok_jacc = _jaccard(set(a1.split()), set(a2.split()))

        exact_name = float(bool(n1) and n1 == n2)
        exact_addr = float(bool(a1) and a1 == a2)

        num1 = set(re.findall(r"\d+", a1))
        num2 = set(re.findall(r"\d+", a2))
        if num1 and num2:
            addr_num_match = 1.0 if num1 == num2 else (0.5 if (num1 & num2) else 0.0)
        else:
            addr_num_match = 0.25  # unknown — neither confirms nor denies

        country_match = 1.0 if (c1 and c2 and c1 == c2) else 0.0

        len1, len2 = len(n1), len(n2)
        name_len_ratio = (min(len1, len2) / max(len1, len2)) if max(len1, len2) > 0 else 0.0
        name_prefix4_match = float(n1[:4] == n2[:4] and len(n1) >= 4 and len(n2) >= 4)

        missing_addr = float(not a1 or not a2)
        missing_name = float(not n1 or not n2)

        rows.append((
            s1id, cid, name_char_sim, name_tok_jacc, addr_char_sim, addr_tok_jacc,
            exact_name, exact_addr, addr_num_match, country_match, name_len_ratio,
            name_prefix4_match, missing_addr, missing_name, float(num_keys), float(min_freq),
        ))

    cols = [
        "s1_id", "candidate_id", "name_char_sim", "name_tok_jacc", "addr_char_sim", "addr_tok_jacc",
        "exact_name", "exact_addr", "addr_num_match", "country_match", "name_len_ratio",
        "name_prefix4_match", "missing_addr", "missing_name", "num_blocking_keys", "min_bucket_freq",
    ]
    return pd.DataFrame(rows, columns=cols)


FEATURE_COLS = [
    "name_char_sim", "name_tok_jacc", "addr_char_sim", "addr_tok_jacc",
    "exact_name", "exact_addr", "addr_num_match", "country_match", "name_len_ratio",
    "name_prefix4_match", "missing_addr", "missing_name", "num_blocking_keys", "min_bucket_freq",
]


def build_lookup(df):
    out = {}
    for eid, name, addr, country in zip(df["entity_id"], df["business_name"], df["business_address"], df["country"]):
        out[eid] = (normalize_business_name(name), normalize_address(addr), (country or "").strip().lower())
    return out


t_feat = time.time()
s1_lookup = build_lookup(S1)
s2_lookup = build_lookup(S2)
s3_lookup = build_lookup(S3)

feat_s2 = build_feature_table(s2_candidates, s1_lookup, s2_lookup, "S2")
feat_s3 = build_feature_table(s3_candidates, s1_lookup, s3_lookup, "S3")
features = pd.concat([feat_s2, feat_s3], ignore_index=True)
del feat_s2, feat_s3
gc.collect()

pair_key = features["s1_id"].values + "|" + features["candidate_id"].values
features["label"] = np.fromiter((p in GT_PAIR_SET for p in pair_key), dtype=np.int8, count=len(pair_key))

print(f"Feature table: {features.shape}, built in {time.time()-t_feat:.1f}s")
print(features["label"].value_counts())
resource_snapshot("after feature engineering")


## 12. Training / validation split

An **S1-aware** split: whole Source-1 entities (and all of their candidate pairs) go entirely to
train or entirely to validation, so no candidate pair leaks information about its own S1 entity
across the split. Singleton S1 entities (zero true matches) are kept in both splits in proportion,
since correctly predicting "no match" is exactly what the F0.5 metric rewards.


In [ ]:

rng = np.random.RandomState(RANDOM_SEED)
all_s1_ids = S1["entity_id"].to_numpy().copy()
rng.shuffle(all_s1_ids)
n_val = int(VALIDATION_FRACTION * len(all_s1_ids))
val_s1_ids = set(all_s1_ids[:n_val])
train_s1_ids = set(all_s1_ids[n_val:])

train_features = features[features["s1_id"].isin(train_s1_ids)]
val_features = features[features["s1_id"].isin(val_s1_ids)]

print(f"Train S1 entities : {len(train_s1_ids):,}  ({len(train_features):,} candidate pairs, "
      f"{train_features['label'].sum():,} positive)")
print(f"Val   S1 entities : {len(val_s1_ids):,}  ({len(val_features):,} candidate pairs, "
      f"{val_features['label'].sum():,} positive)")


## 13. Final matching model

LightGBM (MIT-licensed GBDT, a few hundred KB of trees — trivially under the competition's
8-billion-parameter cap) if available, otherwise scikit-learn's `HistGradientBoostingClassifier`
(also MIT-licensed) as a drop-in fallback. The model scores every candidate pair; Section 14 turns
those scores into final yes/no matches via a validation-tuned decision policy — the model itself
never directly decides "is this a match".


In [ ]:

t_train = time.time()

if HAVE_LIGHTGBM:
    train_set = lgb.Dataset(train_features[FEATURE_COLS], label=train_features["label"])
    val_set = lgb.Dataset(val_features[FEATURE_COLS], label=val_features["label"], reference=train_set)
    lgb_params = {
        "objective": "binary",
        "metric": "auc",
        "num_leaves": 31,
        "max_depth": 6,
        "learning_rate": 0.05,
        "min_data_in_leaf": 50,
        "verbose": -1,
        "seed": RANDOM_SEED,
    }
    model = lgb.train(
        lgb_params, train_set, num_boost_round=400,
        valid_sets=[val_set],
        callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(0)],
    )
    predict_proba = lambda df: model.predict(df[FEATURE_COLS], num_iteration=model.best_iteration)
    print(f"LightGBM trained: best_iteration={model.best_iteration}")
else:
    model = HistGradientBoostingClassifier(
        max_iter=300, max_depth=6, learning_rate=0.06,
        l2_regularization=1.0, random_state=RANDOM_SEED,
        early_stopping=True, validation_fraction=0.15,
    )
    model.fit(train_features[FEATURE_COLS], train_features["label"])
    predict_proba = lambda df: model.predict_proba(df[FEATURE_COLS])[:, 1]
    print("HistGradientBoostingClassifier trained (LightGBM fallback).")

val_features = val_features.copy()
val_features["score"] = predict_proba(val_features)
print(f"Model training time: {time.time()-t_train:.1f}s")

from sklearn.metrics import roc_auc_score
try:
    auc = roc_auc_score(val_features["label"], val_features["score"])
    print(f"Validation pair-level ROC-AUC: {auc:.4f}")
except ValueError:
    pass


## 14. Final matching decision & singleton handling

The model outputs a **score per candidate pair**, not a match/no-match decision. The decision policy
is a single probability threshold, chosen on the validation split to maximize the competition's own
macro-averaged F0.5 — deliberately precision-heavy, since F0.5 weights precision 2x over recall.

An S1 entity with **no candidate pair scoring above the threshold predicts an empty match list**
(a correct singleton prediction scores a full 1.0 under F0.5), so singletons are never forced to
receive a match. No maximum-matches-per-S1 cap is imposed — the threshold alone decides — since the
challenge allows an S1 entity to match zero, one, or many records.


In [ ]:

def macro_f05(scored_df, threshold, s1_id_universe, ground_truth):
    """
    Macro-averaged F0.5 over every S1 id in s1_id_universe (not just ones with candidates),
    matching the competition's own definition: a correctly-predicted singleton scores 1.0.
    """
    preds = (
        scored_df.loc[scored_df["score"] >= threshold]
        .groupby("s1_id")["candidate_id"].apply(set).to_dict()
    )
    scores = []
    for s1id in s1_id_universe:
        pred = preds.get(s1id, set())
        true = ground_truth.get(s1id, {"all": set()})["all"]
        if not pred and not true:
            scores.append(1.0)
        elif not pred and true:
            scores.append(0.0)
        elif pred and not true:
            scores.append(0.0)
        else:
            tp = len(pred & true)
            precision = tp / len(pred)
            recall = tp / len(true)
            denom = 0.25 * precision + recall
            scores.append((1.25 * precision * recall) / denom if denom > 0 else 0.0)
    return float(np.mean(scores)), scores


val_s1_universe = [s for s in S1["entity_id"] if s in val_s1_ids]

threshold_grid = np.round(np.arange(0.05, 0.96, 0.025), 3)
best_threshold, best_f05 = 0.5, -1.0
threshold_curve = []
for th in threshold_grid:
    f05, _ = macro_f05(val_features, th, val_s1_universe, GROUND_TRUTH)
    threshold_curve.append((th, f05))
    if f05 > best_f05:
        best_f05, best_threshold = f05, th

print("Threshold search (validation macro F0.5):")
for th, f05 in threshold_curve:
    marker = "  <-- best" if th == best_threshold else ""
    print(f"  threshold={th:.3f}  macro F0.5={f05:.4f}{marker}")

DECISION_THRESHOLD = float(best_threshold)
print(f"\nSelected decision threshold: {DECISION_THRESHOLD}")


## 15. F0.5 validation report

In [ ]:

final_f05, per_entity_scores = macro_f05(val_features, DECISION_THRESHOLD, val_s1_universe, GROUND_TRUTH)

val_preds = (
    val_features.loc[val_features["score"] >= DECISION_THRESHOLD]
    .groupby("s1_id")["candidate_id"].apply(set).to_dict()
)
tp = fp = fn = 0
n_true_singletons = n_pred_singletons_correct = 0
for s1id in val_s1_universe:
    pred = val_preds.get(s1id, set())
    true = GROUND_TRUTH.get(s1id, {"all": set()})["all"]
    tp += len(pred & true)
    fp += len(pred - true)
    fn += len(true - pred)
    if not true:
        n_true_singletons += 1
        if not pred:
            n_pred_singletons_correct += 1

micro_precision = tp / (tp + fp) if (tp + fp) else 0.0
micro_recall = tp / (tp + fn) if (tp + fn) else 0.0

print("=" * 60)
print("FINAL MATCHING VALIDATION (S1-aware held-out split)")
print("=" * 60)
print(f"Validation S1 entities        : {len(val_s1_universe):,}")
print(f"Decision threshold            : {DECISION_THRESHOLD}")
print(f"Macro F0.5 (competition metric): {final_f05:.4f}")
print(f"Micro precision                : {micro_precision:.4f}")
print(f"Micro recall                   : {micro_recall:.4f}")
print(f"True singletons in val split   : {n_true_singletons:,}")
print(f"  correctly predicted empty    : {n_pred_singletons_correct:,} "
      f"({n_pred_singletons_correct/n_true_singletons*100 if n_true_singletons else 0:.1f}%)")
print("=" * 60)


## 16. 10K benchmark report
The exact report format requested, built entirely from values measured above (nothing here is hard-coded).

In [ ]:

_disk_total, _disk_used, _disk_free = shutil.disk_usage(".")
_ram_mb = psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2) if HAVE_PSUTIL else float("nan")
_elapsed = time.time() - RUN_START

report_10k = f"""
==================================================
10K REFERENCE VALIDATION
==================================================
S1 rows: {len(S1):,}
S2 rows: {len(S2):,}
S3 rows: {len(S3):,}

S2 true matches: {baseline_recall_report['s2_actual']:,}
S2 found: {baseline_recall_report['s2_found']:,}
S2 missed: {baseline_recall_report['s2_actual'] - baseline_recall_report['s2_found']:,}
S2 candidate recall: {baseline_recall_report['s2_recall']*100:.2f}%

S3 true matches: {baseline_recall_report['s3_actual']:,}
S3 found: {baseline_recall_report['s3_found']:,}
S3 missed: {baseline_recall_report['s3_actual'] - baseline_recall_report['s3_found']:,}
S3 candidate recall: {baseline_recall_report['s3_recall']*100:.2f}%

Combined candidate recall: {baseline_recall_report['comb_recall']*100:.2f}%

Candidate pairs: {len(s2_candidates) + len(s3_candidates):,}
Candidates / S1: {baseline_recall_report['avg_candidates_per_s1']:.2f}

Final validation:
Precision: {micro_precision:.4f}
Recall: {micro_recall:.4f}
F0.5: {final_f05:.4f}

Runtime: {_elapsed:.1f}s
Peak RAM: {_ram_mb:,.1f} MB
Disk used: {_disk_used/1024**3:,.2f} GB / {_disk_total/1024**3:,.2f} GB
==================================================
"""
print(report_10k)
(REPORTS_DIR / "10k_benchmark_report.txt").write_text(report_10k, encoding="utf-8")
print(f"Saved: {REPORTS_DIR / '10k_benchmark_report.txt'}")

comparison = f"""
Reference implementation (amazon_ml_solution_FINAL, output/pipeline_report_train_10k.txt):
  S2 recall 98.26% | S3 recall 97.68% | Combined 97.96% | avg candidates/S1 254.19 | runtime ~1565.7s

This notebook (measured above):
  S2 recall {baseline_recall_report['s2_recall']*100:.2f}% | S3 recall {baseline_recall_report['s3_recall']*100:.2f}% | \
Combined {baseline_recall_report['comb_recall']*100:.2f}% | avg candidates/S1 {baseline_recall_report['avg_candidates_per_s1']:.2f} \
| candidate-generation runtime ~{_elapsed:.0f}s end-to-end (incl. feature engineering, model training, F0.5 validation)
"""
print(comparison)


## 17. TEST data discovery

Looks for `test_source1.tsv` / `test_source2.tsv` / `test_source3.tsv` anywhere under the same search
roots as the TRAIN files. **This conversation did not attach TEST files**, so at the point this
notebook was produced, submission generation is wired up but not fabricated — no
`matching_results.tsv` is invented from data that doesn't exist. Attach the official TEST files as a
Kaggle dataset input (or drop them next to the TRAIN files locally) and re-run from this cell down;
everything below auto-detects them.


In [ ]:

TEST_FILES = {
    "s1": "test_source1.tsv",
    "s2": "test_source2.tsv",
    "s3": "test_source3.tsv",
}
TEST_PATHS = {k: find_file(v, SEARCH_ROOTS) for k, v in TEST_FILES.items()}
TEST_AVAILABLE = all(v is not None for v in TEST_PATHS.values())

print("Discovered TEST files:")
for k, v in TEST_PATHS.items():
    print(f"  {k:>3}: {v}")
print()
if TEST_AVAILABLE:
    print("TEST data found -> Sections 18-23 will generate the official submission outputs.")
else:
    print("TEST data NOT found -> Sections 18-23 will define generate_submission() but will "
          "NOT run it or fabricate output files. Re-run once test_source{1,2,3}.tsv are attached.")


## 18–21. TEST candidate generation, TEST matching, and official output generation

One function, `generate_submission()`, runs the *exact same* normalization / blocking / feature /
model code defined above against TEST data and writes both official output files:

- `output/candidate_pairs.tsv` — every candidate this pipeline considered, before the final threshold
  narrows it down (one row per TEST S1 entity; empty when blocking found nothing).
- `output/matching_results.tsv` — the final matches after applying `DECISION_THRESHOLD` (one row per
  TEST S1 entity; empty `matched_entity_ids` for a predicted singleton).

The model, blocking keys, frequency caps and decision threshold are all the ones already fit/tuned on
TRAIN above — TEST ground truth does not exist and is never used for anything here.


In [ ]:

def _format_id_list(ids):
    """No brackets, no quotes, no 'None'/'NaN', comma-separated, de-duplicated, order-preserving."""
    seen = []
    seen_set = set()
    for i in ids:
        if i and i not in seen_set:
            seen_set.add(i)
            seen.append(i)
    return ",".join(seen)


def generate_submission(test_s1_path, test_s2_path, test_s3_path, output_dir=OUTPUT_DIR,
                         model=None, predict_fn=None, threshold=None, chunk_size=S1_CHUNK_SIZE):
    """
    Runs the full pipeline (blocking -> candidate generation -> features -> scoring -> decision)
    against TEST data and writes output/candidate_pairs.tsv and output/matching_results.tsv.

    Uses the SAME functions defined earlier in this notebook (make_all_keys, build_filtered_index,
    generate_candidates, build_feature_table, ...) so TRAIN and TEST always go through identical code.
    """
    threshold = DECISION_THRESHOLD if threshold is None else threshold
    predict_fn = predict_proba if predict_fn is None else predict_fn

    print("Loading TEST data...")
    test_s1 = read_tsv(test_s1_path, limit=TEST_S1_LIMIT)
    test_s2 = read_tsv(test_s2_path)
    test_s3 = read_tsv(test_s3_path)
    print(f"  TEST S1: {len(test_s1):,}  TEST S2: {len(test_s2):,}  TEST S3: {len(test_s3):,}")

    print("Building TEST S2/S3 blocking indexes...")
    t_s2_key = build_key_table(test_s2)
    t_s3_key = build_key_table(test_s3)
    t_s2_index, _ = build_filtered_index(t_s2_key)
    t_s3_index, _ = build_filtered_index(t_s3_key)

    print("Generating TEST candidates...")
    t_s2_cands = generate_candidates(test_s1, t_s2_index, "S2", chunk_size=chunk_size)
    t_s3_cands = generate_candidates(test_s1, t_s3_index, "S3", chunk_size=chunk_size)
    print(f"  S2 candidate pairs: {len(t_s2_cands):,}  S3 candidate pairs: {len(t_s3_cands):,}")

    print("Building TEST features...")
    test_s1_lookup = build_lookup(test_s1)
    test_s2_lookup = build_lookup(test_s2)
    test_s3_lookup = build_lookup(test_s3)
    t_feat_s2 = build_feature_table(t_s2_cands, test_s1_lookup, test_s2_lookup, "S2")
    t_feat_s3 = build_feature_table(t_s3_cands, test_s1_lookup, test_s3_lookup, "S3")
    test_features = pd.concat([t_feat_s2, t_feat_s3], ignore_index=True)
    del t_feat_s2, t_feat_s3
    gc.collect()

    print("Scoring TEST candidates...")
    test_features["score"] = predict_fn(test_features)

    all_test_s1_ids = test_s1["entity_id"].tolist()

    cand_map = test_features.groupby("s1_id")["candidate_id"].apply(list).to_dict()
    match_map = (
        test_features.loc[test_features["score"] >= threshold]
        .groupby("s1_id")["candidate_id"].apply(list).to_dict()
    )

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    cand_rows = [{"source1_entity_id": s1id, "candidate_entity_ids": _format_id_list(cand_map.get(s1id, []))}
                 for s1id in all_test_s1_ids]
    match_rows = [{"source1_entity_id": s1id, "matched_entity_ids": _format_id_list(match_map.get(s1id, []))}
                  for s1id in all_test_s1_ids]

    cand_out = pd.DataFrame(cand_rows, columns=["source1_entity_id", "candidate_entity_ids"])
    match_out = pd.DataFrame(match_rows, columns=["source1_entity_id", "matched_entity_ids"])

    cand_path = output_dir / "candidate_pairs.tsv"
    match_path = output_dir / "matching_results.tsv"
    cand_out.to_csv(cand_path, sep="\t", index=False)
    match_out.to_csv(match_path, sep="\t", index=False)

    print(f"Wrote {cand_path}  ({len(cand_out):,} rows)")
    print(f"Wrote {match_path}  ({len(match_out):,} rows)")

    n_matched = sum(1 for s1id in all_test_s1_ids if match_map.get(s1id))
    n_singleton = len(all_test_s1_ids) - n_matched
    print(f"Predicted matches for {n_matched:,} S1 entities; "
          f"predicted singleton (no match) for {n_singleton:,} S1 entities.")

    return cand_path, match_path


if TEST_AVAILABLE:
    candidate_pairs_path, matching_results_path = generate_submission(
        TEST_PATHS["s1"], TEST_PATHS["s2"], TEST_PATHS["s3"],
    )
else:
    print("generate_submission() is defined and ready, but was NOT run: no TEST files were found "
          "under the search roots. Attach test_source1/2/3.tsv and re-run this cell (or call "
          "generate_submission(path1, path2, path3) directly) to produce the official submission files.")


## 22. Submission validator

The official `utils/validate_submission.py` was not provided with this conversation's uploads, so
this notebook ships an equivalent, stdlib-only validator implementing every rule from the problem
statement (one row per TEST S1 entity, no duplicate rows/IDs, S2/S3-only IDs that exist in the TEST
set, matches ⊆ candidates, correct TSV columns/delimiter, empty singleton rows allowed). **If the
official `utils/validate_submission.py` is available, drop it next to this notebook and prefer it —
this fallback is provided so the workflow is still self-checking without it.**


In [ ]:

def validate_submission(matching_path, candidate_path, test_s1_path, test_s2_path, test_s3_path):
    issues = []

    test_s1_ids = set(read_tsv(test_s1_path)["entity_id"])
    valid_other_ids = set(read_tsv(test_s2_path)["entity_id"]) | set(read_tsv(test_s3_path)["entity_id"])

    def _check(path, id_col):
        df = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False)
        local_issues = []
        if list(df.columns[:1]) != ["source1_entity_id"]:
            local_issues.append(f"{path}: first column must be 'source1_entity_id', got {list(df.columns)}")
        if df["source1_entity_id"].duplicated().any():
            dupes = df.loc[df["source1_entity_id"].duplicated(), "source1_entity_id"].unique()[:5]
            local_issues.append(f"{path}: duplicate source1_entity_id rows, e.g. {list(dupes)}")
        present = set(df["source1_entity_id"])
        missing = test_s1_ids - present
        extra = present - test_s1_ids
        if missing:
            local_issues.append(f"{path}: {len(missing)} TEST S1 entities missing, e.g. {list(missing)[:5]}")
        if extra:
            local_issues.append(f"{path}: {len(extra)} rows reference non-TEST S1 ids, e.g. {list(extra)[:5]}")
        id_lists = {}
        for s1id, raw in zip(df["source1_entity_id"], df[id_col]):
            raw = (raw or "").strip()
            ids = [x for x in raw.split(",") if x] if raw else []
            if len(ids) != len(set(ids)):
                local_issues.append(f"{path}: duplicate IDs within list for {s1id}: {ids}")
            bad = [i for i in ids if not (i.startswith("S2-") or i.startswith("S3-")) or i not in valid_other_ids]
            if bad:
                local_issues.append(f"{path}: invalid/self/unknown IDs for {s1id}: {bad[:5]}")
            id_lists[s1id] = set(ids)
        return local_issues, id_lists

    match_issues, match_lists = _check(matching_path, "matched_entity_ids")
    cand_issues, cand_lists = _check(candidate_path, "candidate_entity_ids")
    issues.extend(match_issues)
    issues.extend(cand_issues)

    for s1id, matched in match_lists.items():
        candidates = cand_lists.get(s1id, set())
        if not matched.issubset(candidates):
            issues.append(f"{s1id}: matched ids not a subset of candidate ids "
                           f"(matched-only: {matched - candidates})")

    if issues:
        print(f"FAIL — {len(issues)} issue(s) found:")
        for i, msg in enumerate(issues, 1):
            print(f"  {i}. {msg}")
        return False
    print("PASS — submission files satisfy all checked rules.")
    return True


if TEST_AVAILABLE:
    validation_passed = validate_submission(
        matching_results_path, candidate_pairs_path,
        TEST_PATHS["s1"], TEST_PATHS["s2"], TEST_PATHS["s3"],
    )
else:
    print("Skipping validation: no TEST outputs were generated (see Section 17/18-21).")
    validation_passed = None


## 23. Final Kaggle resource check

If this notebook is being used as the validated 10K development artifact, the run is complete when the F0.5 report prints and `reports/10k_benchmark_report.txt` exists. Do not switch `S1_LIMIT` to `None` here. The full leaderboard pipeline requires a separately profiled, memory/disk-aware implementation.

In [ ]:
total, used, free = shutil.disk_usage('.')
print(f'Working disk used: {used/1024**3:.2f} GB / {total/1024**3:.2f} GB')
if used/1024**3 > 19.0:
    print('WARNING: working disk is above 19 GB; stop before generating additional artifacts.')
else:
    print('Kaggle working-disk guard: OK for the 20 GB target.')
print('Notebook completed without launching an unbounded full-scale run.')
